In [3]:
import re
import time
import datetime
import logging
import argparse
from pathlib import Path
from dataclasses import dataclass

import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── HTTP session ──────────────────────────────────────────────────────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "LagoonCinemaScraper/1.0 "
        "(historical-research; polite delay between requests)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})
DELAY = 1.2  # seconds between requests

CINEMACLOCK_URL      = "https://www.cinemaclock.com/movie-theaters/lagoon-cinema"
RACKET_MOVIES_BASE   = "https://racketmn.com/category/movies"
LANDMARK_SECRET_URL  = "https://www.landmarktheatres.com/secret-movie/"


# ─────────────────────────────────────────────────────────────────────────────
# HTTP HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def get(url: str, **kwargs) -> requests.Response:
    """GET with 3 retries, back-off, and polite delay."""
    for attempt in range(3):
        try:
            r = SESSION.get(url, timeout=20, **kwargs)
            r.raise_for_status()
            time.sleep(DELAY)
            return r
        except requests.RequestException as exc:
            log.warning("Attempt %d failed for %s : %s", attempt + 1, url, exc)
            time.sleep(DELAY * (attempt + 2))
    raise RuntimeError(f"Failed to fetch {url} after 3 attempts")


def soup(url: str, **kwargs) -> BeautifulSoup:
    return BeautifulSoup(get(url, **kwargs).text, "html.parser")


# ─────────────────────────────────────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class LagoonScreening:
    date:            str   # YYYY-MM-DD or descriptive string
    day:             str   # Mon / Tue / …
    film_title:      str
    film_year:       str
    distributor:     str   # or price for special screenings
    type_:           str   # FIRST_RUN / SECRET_MOVIE / REVIVAL / CONCERT_FILM / LIVE_EVENT / SPECIAL
    wknd_showtimes:  str   # Fri/Sat/Sun times, space-separated
    wkday_showtimes: str   # Mon–Thu times, space-separated
    screens:         str   # "1", "2", "3-4", etc.
    genre:           str
    run_weeks:       str
    confidence:      str   # "✅ Confirmed" / "⚠️ Likely"
    notes:           str
    source_url:      str = ""


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

_DOW = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}

_TIME_RE = re.compile(
    r"\b(\d{1,2}:\d{2}\s*(?:am|pm)|"
    r"\d{1,2}\s*(?:a\.?m\.?|p\.?m\.?)|"
    r"noon|midnight)\b",
    re.IGNORECASE,
)

_GENRE_RE = re.compile(
    r"\b(Drama|Comedy|Horror|Action|Animation|Documentary|Thriller|"
    r"Sci-Fi|Romance|Musical|Western|Mystery|Fantasy|Biography|"
    r"Concert Film|Opera|Ballet)\b",
    re.IGNORECASE,
)

_TYPE_KEYWORDS = {
    "SECRET_MOVIE":  ["secret movie", "secret"],
    "CONCERT_FILM":  ["concert film", "concert", "live from", "beach party", "dreamworld"],
    "LIVE_EVENT":    ["live at the met", "opera", "ballet", "royal ballet"],
    "REVIVAL":       ["revival", "re-release", "restored", "classic"],
    "ADVANCE":       ["advance", "preview", "q&a", "with cast"],
}


def classify_type(title: str, notes: str = "", price: str = "") -> str:
    combined = (title + " " + notes).lower()
    for type_, keywords in _TYPE_KEYWORDS.items():
        if any(k in combined for k in keywords):
            return type_
    if price == "$5":
        return "SECRET_MOVIE"
    return "FIRST_RUN"


def normalise_time(raw: str) -> str:
    """Turn 'noon', '7pm', '7:30 p.m.' into clean '7:30 PM' strings."""
    t = raw.strip().upper().replace(".", "").replace("A M", "AM").replace("P M", "PM")
    if t == "NOON":
        return "12:00 PM"
    if t == "MIDNIGHT":
        return "12:00 AM"
    if ":" not in t:
        hr = int(re.search(r"\d+", t).group())
        suffix = "PM" if "PM" in t else "AM"
        return f"{hr}:00 {suffix}"
    # already has colon — ensure no space between digits and AM/PM
    t = re.sub(r"(\d)\s+(AM|PM)", r"\1 \2", t)
    return t


def extract_times_from_text(text: str) -> list[str]:
    """Return all time strings found in `text`, normalised."""
    return [normalise_time(m.group(1)) for m in _TIME_RE.finditer(text)]


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 1 — CINEMACLOCK  (live current-week showtimes)
# ─────────────────────────────────────────────────────────────────────────────

def scrape_cinemaclock() -> list[LagoonScreening]:
    """
    Parse the CinemaClock page for Lagoon Cinema.

    Page structure (2026):  for each film there is an <h3> heading, a metadata
    paragraph (rating, year, runtime, genre), and showtime text blocks with lines
    like "Today Apr 18   1:40  5:00  7:30" and "Mon Apr 20   4:30  7:25".

    Returns one LagoonScreening per film, split into wknd / wkday showtime strings.
    """
    log.info("[CinemaClock] Fetching: %s", CINEMACLOCK_URL)
    try:
        page = soup(CINEMACLOCK_URL)
    except RuntimeError as exc:
        log.warning("  CinemaClock fetch failed: %s", exc)
        return []

    today      = datetime.date.today()
    screenings: list[LagoonScreening] = []

    # Each film is introduced by an <h3>; we walk the DOM from heading to heading
    headings = page.find_all("h3")

    for h3 in headings:
        film_title = h3.get_text(strip=True)
        if not film_title or len(film_title) < 2:
            continue

        meta_text  = ""
        day_lines: list[tuple[str, list[str]]] = []   # (day_label, [times])

        # Collect sibling nodes until next h3
        node = h3.find_next_sibling()
        while node and node.name != "h3":
            text = node.get_text(" ", strip=True)

            # metadata line: contains rating + year
            if re.search(r"\b(PG|R|G|NR|PG-13)\b", text) and re.search(r"\b20\d{2}\b", text):
                meta_text = text

            # showtime lines: "Today Apr 18 1:40 5:00 7:30"
            day_m = re.search(
                r"\b(Today|Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b.*?"
                r"((?:\d{1,2}:\d{2}\s*)+)",
                text, re.IGNORECASE,
            )
            if day_m:
                day_label = day_m.group(1).strip().lower()
                raw_times  = day_m.group(2).strip()
                times = re.findall(r"\d{1,2}:\d{2}", raw_times)
                # assign AM/PM heuristically (theater runs ~noon–midnight)
                fmt_times = [
                    f"{t} PM" if int(t.split(":")[0]) >= 11 or int(t.split(":")[0]) < 1
                    else f"{t} PM"   # all theater times are PM in practice
                    for t in times
                ]
                if fmt_times:
                    day_lines.append((day_label, fmt_times))

            node = node.find_next_sibling()

        if not day_lines:
            continue

        # extract year + genre from metadata
        yr_m  = re.search(r"\b(20\d{2})\b", meta_text)
        film_year = yr_m.group(1) if yr_m else str(today.year)
        genre_m   = _GENRE_RE.search(meta_text)
        genre     = genre_m.group(1).title() if genre_m else ""

        # split into weekend vs weekday
        wknd_slots:  list[str] = []
        wkday_slots: list[str] = []
        for day_label, times in day_lines:
            times_str = "  ".join(times)
            if day_label in ("fri", "sat", "sun", "today"):
                wknd_slots.append(times_str)
            else:
                wkday_slots.append(times_str)

        # use first occurrence of each bucket (opening day pattern)
        wknd  = wknd_slots[0]  if wknd_slots  else (wkday_slots[0] if wkday_slots else "")
        wkday = wkday_slots[0] if wkday_slots else wknd

        type_ = classify_type(film_title)

        screenings.append(LagoonScreening(
            date=today.isoformat(),
            day=_DOW[today.weekday()],
            film_title=film_title,
            film_year=film_year,
            distributor="",
            type_=type_,
            wknd_showtimes=wknd,
            wkday_showtimes=wkday,
            screens="1" if type_ != "FIRST_RUN" else "2",
            genre=genre,
            run_weeks="1",
            confidence="✅ Confirmed",
            notes=f"CinemaClock live data {today.isoformat()}",
            source_url=CINEMACLOCK_URL,
        ))

    log.info("[CinemaClock] %d films found", len(screenings))
    return screenings


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 2 — RACKET MN ARCHIVE  (historical Lagoon mentions)
# ─────────────────────────────────────────────────────────────────────────────

def get_racket_article_urls(max_pages: int = 150) -> list[str]:
    """
    Walk the Racket MN /category/movies archive and return all article URLs.
    Pagination uses a GraphQL cursor encoded in ?after=BASE64 query params.
    """
    log.info("[Racket] Collecting article URLs (max %d index pages) …", max_pages)
    urls:  list[str] = []
    seen:  set[str]  = set()
    current = RACKET_MOVIES_BASE

    for page_num in range(1, max_pages + 1):
        log.info("  Index page %d: %s", page_num, current[:90])
        try:
            page = soup(current)
        except RuntimeError as exc:
            log.warning("  Stopped: %s", exc)
            break

        new = 0
        # Article links: typically wrapped in <a> inside <article> or heading tags
        for a in page.select(
            "article a[href], h2 a[href], h3 a[href], "
            ".post-title a[href], .entry-title a[href]"
        ):
            href = a.get("href", "")
            # Only keep article slugs (long path segments, not category/tag pages)
            if (href
                    and "racketmn.com" in href
                    and href not in seen
                    and len(href.split("/")[-2]) > 8):
                seen.add(href)
                urls.append(href)
                new += 1

        if new == 0:
            log.info("  No new articles on page %d — stopping", page_num)
            break

        # Find "Older" pagination link
        older = (
            page.find("a", string=re.compile(r"older", re.I))
            or page.find("a", rel="next")
        )
        if older and older.get("href"):
            nxt = older["href"]
            current = nxt if nxt.startswith("http") else "https://racketmn.com" + nxt
        else:
            break

    log.info("[Racket] Total article URLs: %d", len(urls))
    return urls


def parse_racket_article(url: str) -> list[LagoonScreening]:
    """
    Fetch and parse one Racket MN weekly listing article.

    Searches every sentence that mentions "Lagoon" and extracts:
      - film title  (text before "Lagoon Cinema")
      - film year   (in parentheses)
      - showtime    (e.g. "7 p.m.", "7:30 p.m.", "Noon")
      - price       (e.g. "$8", "$12.25")
      - article publish date (from <time> or URL slug)
    """
    try:
        page = soup(url)
    except RuntimeError as exc:
        log.debug("  Skip %s : %s", url, exc)
        return []

    # Publish date
    pub_date = ""
    time_el  = page.find("time")
    if time_el:
        raw = time_el.get("datetime", "") or time_el.get_text(strip=True)
        m = re.search(r"(\d{4}-\d{2}-\d{2})", raw)
        if m:
            pub_date = m.group(1)
    if not pub_date:
        m = re.search(r"(\d{4})[/-](\d{2})[/-](\d{2})", url)
        if m:
            pub_date = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

    content = page.select_one(
        "article .entry-content, article .post-content, article, main"
    )
    if not content:
        return []

    full_text = content.get_text("\n", strip=True)
    screenings: list[LagoonScreening] = []

    # Split into paragraphs; only process those mentioning Lagoon
    for para in re.split(r"\n{2,}", full_text):
        if "lagoon" not in para.lower():
            continue

        # Further split into sentences
        for sent in re.split(r"(?<=[.!?])\s+", para):
            if "lagoon" not in sent.lower():
                continue

            # Film title = text before the first "Lagoon" occurrence
            lagoon_pos = sent.lower().find("lagoon")
            raw_before = sent[:lagoon_pos].strip()
            if not raw_before:
                continue

            # Remove leading junk from a prior sentence fragment
            raw_before = re.sub(r"""^[^A-Za-z0-9"'(]+""", "", raw_before).strip()

            yr_m = re.search(r"\((\d{4})\)", raw_before)
            film_year  = yr_m.group(1) if yr_m else ""
            film_title = re.sub(r"\s*\(\d{4}\)\s*", "", raw_before).strip()
            # strip trailing description after em-dash or pipe
            film_title = re.sub(r"\s*[|—–\-]{1,2}.*$", "", film_title).strip()

            if not film_title or len(film_title) < 3:
                continue

            # Text after "Lagoon [Cinema]"
            after = sent[lagoon_pos + 6:]

            # showtime
            times = extract_times_from_text(after) or extract_times_from_text(sent)
            showtime = times[0] if times else ""

            # price
            price_m = re.search(r"\$(\d+(?:\.\d{1,2})?)", after)
            price   = f"${price_m.group(1)}" if price_m else ""

            # classify
            type_ = classify_type(film_title, sent, price)

            # day-of-week
            dow = ""
            if pub_date:
                try:
                    d   = datetime.date.fromisoformat(pub_date)
                    dow = _DOW[d.weekday()]
                except ValueError:
                    pass

            screenings.append(LagoonScreening(
                date=pub_date,
                day=dow,
                film_title=film_title,
                film_year=film_year,
                distributor=price,
                type_=type_,
                wknd_showtimes=showtime,
                wkday_showtimes=showtime,
                screens="1" if type_ in ("REVIVAL", "CONCERT_FILM", "SECRET_MOVIE", "LIVE_EVENT") else "2",
                genre="",
                run_weeks="1",
                confidence="✅ Confirmed",
                notes=f'Racket MN: "{sent.strip()[:120]}"',
                source_url=url,
            ))

    log.debug("[Racket] %s → %d Lagoon mentions", url, len(screenings))
    return screenings


# ─────────────────────────────────────────────────────────────────────────────
# SOURCE 3 — LANDMARK SECRET MOVIE PAGE
# ─────────────────────────────────────────────────────────────────────────────

def scrape_landmark_secret_movies() -> list[LagoonScreening]:
    """
    Scrape the Landmark Secret Movie "previously shown" archive.
    Lists film titles only — no dates.  Always at 7:00 PM on Mondays.
    """
    log.info("[Landmark] Fetching Secret Movie archive: %s", LANDMARK_SECRET_URL)
    try:
        page = soup(LANDMARK_SECRET_URL)
    except RuntimeError as exc:
        log.warning("  Secret Movie page failed: %s", exc)
        return []

    screenings: list[LagoonScreening] = []
    skip = {"previously shown", "upcoming secret movie", "secret movie", "$5 secret movie", ""}

    for el in page.select("h2, h3, h4, .movie-title, .film-title, figure figcaption"):
        title = el.get_text(strip=True)
        if title.lower() in skip or len(title) < 3:
            continue
        screenings.append(LagoonScreening(
            date="",
            day="Mon",       # Landmark secret movies are always Monday evenings
            film_title=title,
            film_year="",
            distributor="$5",
            type_="SECRET_MOVIE",
            wknd_showtimes="7:00 PM",
            wkday_showtimes="7:00 PM",
            screens="1",
            genre="Secret",
            run_weeks="1",
            confidence="✅ Confirmed",
            notes="Landmark Secret Movie archive — date unknown",
            source_url=LANDMARK_SECRET_URL,
        ))

    log.info("[Landmark] %d secret movies found", len(screenings))
    return screenings


# ─────────────────────────────────────────────────────────────────────────────
# DEDUPLICATION + SORT
# ─────────────────────────────────────────────────────────────────────────────

def _sort_key(s: LagoonScreening) -> datetime.date:
    try:
        return datetime.date.fromisoformat(s.date)
    except ValueError:
        return datetime.date(2099, 1, 1)


def dedupe_and_sort(screenings: list[LagoonScreening]) -> list[LagoonScreening]:
    seen:   set[tuple]          = set()
    unique: list[LagoonScreening] = []
    for s in screenings:
        key = (s.date, s.film_title[:35].lower().strip())
        if key not in seen:
            seen.add(key)
            unique.append(s)
    unique.sort(key=_sort_key)
    return unique


# ─────────────────────────────────────────────────────────────────────────────
# EXCEL OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

_DARK_BLUE = "1F3864"
_MED_BLUE  = "2E75B6"
_TYPE_COLORS = {
    "FIRST_RUN":    "DAEEF3",
    "REVIVAL":      "CFE2F3",
    "SECRET_MOVIE": "C6EFCE",
    "CONCERT_FILM": "FFF2CC",
    "LIVE_EVENT":   "FCE4D6",
    "ADVANCE":      "EAD1DC",
    "SPECIAL":      "D0E0E3",
}
_thin = Side(style="thin", color="BFBFBF")
_BD   = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)


def _hdr(cell, txt: str):
    cell.value     = txt
    cell.font      = Font(bold=True, color="FFFFFF", size=9, name="Arial")
    cell.fill      = PatternFill("solid", start_color=_MED_BLUE)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border    = _BD


def _data(cell, val, bg: str = "DAEEF3", wrap: bool = False, center: bool = False):
    cell.value     = str(val) if val is not None else ""
    cell.font      = Font(size=8, name="Arial")
    cell.fill      = PatternFill("solid", start_color=bg)
    cell.alignment = Alignment(vertical="center", wrap_text=wrap,
                               horizontal="center" if center else "left")
    cell.border    = _BD


def write_xlsx(screenings: list[LagoonScreening], path: str):
    wb = Workbook()
    ws = wb.active
    ws.title = "Lagoon Screenings"

    # ── title row ──────────────────────────────────────────────────────────
    ws.merge_cells("A1:M1")
    t = ws["A1"]
    t.value = (
        "Lagoon Cinema (Landmark Theatres) — Screening Log "
        "| 1320 Lagoon Ave, Minneapolis MN 55408 "
        "| Sources: CinemaClock · Racket MN · Landmark Secret Movie archive"
    )
    t.font      = Font(bold=True, size=11, color="FFFFFF", name="Arial")
    t.fill      = PatternFill("solid", start_color=_DARK_BLUE)
    t.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 20

    # ── legend row ─────────────────────────────────────────────────────────
    ws.merge_cells("A2:M2")
    leg = ws["A2"]
    leg.value = (
        "🟦 FIRST_RUN  🔵 REVIVAL  🟩 SECRET_MOVIE  🟨 CONCERT_FILM  "
        "🟧 LIVE_EVENT  🌸 ADVANCE/SPECIAL  "
        "✅ = confirmed via source  ⚠️ = inferred from Landmark booking patterns"
    )
    leg.font      = Font(italic=True, size=8, color="7F0000", name="Arial")
    leg.fill      = PatternFill("solid", start_color="FCE4D6")
    leg.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    ws.row_dimensions[2].height = 28

    # ── column headers ─────────────────────────────────────────────────────
    cols = [
        ("Date", 13), ("Day", 5), ("Film Title", 38), ("Film Year", 7),
        ("Distrib / Price", 13), ("Type", 15),
        ("Fri/Sat/Sun\nShowtimes", 22), ("Mon–Thu\nShowtimes", 18),
        ("Screens", 7), ("Genre", 15), ("Run (wks)", 8),
        ("Confidence", 10), ("Notes / Source", 44),
    ]
    for i, (label, width) in enumerate(cols, 1):
        _hdr(ws.cell(row=3, column=i), label)
        ws.column_dimensions[get_column_letter(i)].width = width
    ws.row_dimensions[3].height = 26

    # ── data rows ──────────────────────────────────────────────────────────
    for r_idx, s in enumerate(screenings, 4):
        bg = _TYPE_COLORS.get(s.type_, "DAEEF3")
        row_vals = [
            s.date, s.day, s.film_title, s.film_year,
            s.distributor, s.type_,
            s.wknd_showtimes, s.wkday_showtimes,
            s.screens, s.genre, s.run_weeks,
            s.confidence,
            (s.notes + ("  |  " + s.source_url if s.source_url else ""))[:160],
        ]
        for c_idx, val in enumerate(row_vals, 1):
            _data(
                ws.cell(row=r_idx, column=c_idx),
                val, bg=bg,
                wrap=(c_idx in (3, 7, 8, 13)),
                center=(c_idx in (2, 4, 9, 10, 11, 12)),
            )
        ws.row_dimensions[r_idx].height = 15

    ws.freeze_panes = "A4"
    ws.auto_filter.ref = f"A3:M{len(screenings) + 3}"
    wb.save(path)
    log.info("Saved %s  (%d rows)", path, len(screenings))


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Scrape Lagoon Cinema showtimes and write to Excel."
    )
    parser.add_argument("--out-dir",             default=".",   help="Output directory")
    parser.add_argument("--delay",               type=float, default=1.2,
                        help="Seconds between HTTP requests (default: 1.2)")
    parser.add_argument("--max-racket-pages",    type=int,   default=150,
                        help="Max Racket MN archive index pages to walk (default: 150)")
    parser.add_argument("--skip-cinemaclock",    action="store_true")
    parser.add_argument("--skip-racket",         action="store_true")
    parser.add_argument("--skip-secret-movies",  action="store_true")
    args = parser.parse_args()

    global DELAY
    DELAY = args.delay

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    all_screenings: list[LagoonScreening] = []

    # ── 1. CinemaClock (live current-week showtimes) ───────────────────────
    if not args.skip_cinemaclock:
        log.info("=== SOURCE 1: CINEMACLOCK (live) ===")
        all_screenings.extend(scrape_cinemaclock())

    # ── 2. Racket MN archive ───────────────────────────────────────────────
    if not args.skip_racket:
        log.info("=== SOURCE 2: RACKET MN ARCHIVE ===")
        article_urls = get_racket_article_urls(max_pages=args.max_racket_pages)
        for url in article_urls:
            all_screenings.extend(parse_racket_article(url))

    # ── 3. Landmark Secret Movie archive ──────────────────────────────────
    if not args.skip_secret_movies:
        log.info("=== SOURCE 3: LANDMARK SECRET MOVIES ===")
        all_screenings.extend(scrape_landmark_secret_movies())

    # ── Deduplicate, sort, write ────────────────────────────────────────────
    final = dedupe_and_sort(all_screenings)
    log.info("Total unique screenings: %d", len(final))

    out_path = str(out / "lagoon_screenings.xlsx")
    write_xlsx(final, out_path)
    log.info("Done.  Output: %s", out_path)


if __name__ == "__main__":
    main()


'C:\\Users\\S_Ben'